In [1]:
from tqdm.notebook import tqdm
import os,pickle,glob,gc,sys
import polars as pl
import pandas as pd
import numpy as np

In [2]:
train=pd.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/train.parquet')
test=pd.read_parquet('/home/mingyu/Recommand-System/项目/OTTO/data/processData/test.parquet')

In [3]:
train.index=pd.MultiIndex.from_frame(train[['session']])
test.index=pd.MultiIndex.from_frame(test[['session']])

In [4]:
data=pd.concat([train,test])
data.head(),data.shape

(         session      aid          ts  type
 session                                    
 0              0  1517085  1659304800     0
 0              0  1563459  1659304904     0
 0              0  1309446  1659367439     0
 0              0    16246  1659367719     0
 0              0  1781822  1659367871     0,
 (223644219, 4))

In [5]:
from collections import defaultdict,Counter
next_AIDs=defaultdict(Counter)
chunk_size=30_000
max_ts=data['ts'].max()
min_ts=data['ts'].min()
max_ts,min_ts

(np.int32(1662328791), np.int32(1659304800))

In [6]:
sessions=data.session.unique()

In [ ]:
nextID=defaultdict(Counter)

type_weight_multipliers = {0: 1, 1: 6, 2: 3}
for i in range(0,sessions.shape[0],chunk_size):
    # reset_index(drop=True)重建行索引，丢弃旧索引
    current_chunk=data.loc[sessions[i]:sessions[min(sessions.shape[0]-1,i+chunk_size-1)]].reset_index(drop=True)
    # 取出每个session后30个，考虑性能和最近行为
    # as_index保持原有的索引，不用session做索引
    current_chunk=current_chunk.groupby('session',as_index=False).nth(list(range(-30,0))).reset_index(drop=True)

    # 按照session连接，对同一个session内的所有行（不区分属性）做笛卡尔积
    consecutive_AIDs=current_chunk.merge(current_chunk,on='session')
    # 排除自己和自己
    consecutive_AIDs=consecutive_AIDs[consecutive_AIDs.aid_x!=consecutive_AIDs.aid_y]
    # 通过时间戳(>=0)的方式保证两两算一次，不考虑顺序
    consecutive_AIDs['days_elapsed']=(consecutive_AIDs.ts_y-consecutive_AIDs.ts_x)/(24*60*60)
    consecutive_AIDs=consecutive_AIDs[(consecutive_AIDs.days_elapsed>=0) & (consecutive_AIDs.days_elapsed<=1)]
    # 按照时间加权，越新权重越大
    weight=1+3*(consecutive_AIDs.ts_x-min_ts)/(max_ts-min_ts)
    for aid_x,aid_y,type_y in zip(consecutive_AIDs['aid_x'],consecutive_AIDs['aid_y'],consecutive_AIDs['type_y']):
        next_AIDs[aid_x][aid_y]+=weight

# 为每个商品保存分数最大的20个
sub=[]
for aid_x, co_dict in next_AIDs.items():
    for aid_y, _ in co_dict.most_common(20):
        sub.append((aid_x, aid_y))
sub = pd.DataFrame(sub, columns=['aid_x','aid_y'])
sub.to_parquet('/home/mingyu/Recommand-System/项目/OTTO/save/co_visitation_result/click.parquet', index=False)

In [24]:
nextID=defaultdict(Counter)

# 对于cart操作，针对后续操作给定不同的权值
type_weight_multipliers = {0: 1, 1: 6, 2: 3}

for i in range(0,sessions.shape[0],chunk_size):
    # reset_index(drop=True)重建行索引，丢弃旧索引
    current_chunk=data.loc[sessions[i]:sessions[min(sessions.shape[0]-1,i+chunk_size-1)]].reset_index(drop=True)
    # 取出每个session后30个，考虑性能和最近行为
    # as_index保持原有的索引，不用session做索引
    current_chunk=current_chunk.groupby('session',as_index=False).nth(list(range(-30,0))).reset_index(drop=True)

    # 按照session连接，对同一个session内的所有行（不区分属性）做笛卡尔积
    consecutive_AIDs=current_chunk.merge(current_chunk,on='session')

    # 排除自己和自己
    consecutive_AIDs=consecutive_AIDs[consecutive_AIDs.aid_x!=consecutive_AIDs.aid_y]

    # 通过时间戳(>=0)的方式保证两两算一次，不考虑顺序，并取一天内的操作
    consecutive_AIDs['days_elapsed']=(consecutive_AIDs.ts_y-consecutive_AIDs.ts_x)/(24*60*60)
    consecutive_AIDs=consecutive_AIDs[(consecutive_AIDs.days_elapsed>=0) & (consecutive_AIDs.days_elapsed<=1)]

    # 计算共现矩阵
    for aid_x,aid_y,type_y in zip(consecutive_AIDs['aid_x'],consecutive_AIDs['aid_y'],consecutive_AIDs['type_y']):
        next_AIDs[aid_x][aid_y]+=type_weight_multipliers[type_y]

# 保存分数最高的15个
sub=[]
for aid_x, co_dict in next_AIDs.items():
    for aid_y, _ in co_dict.most_common(15):
        sub.append((aid_x, aid_y))
sub = pd.DataFrame(sub, columns=['aid_x','aid_y'])
sub.to_parquet('/home/mingyu/Recommand-System/项目/OTTO/save/co_visitation_result/cart.parquet', index=False)

KeyboardInterrupt: 

In [ ]:
nextID=defaultdict(Counter)

for i in range(0,sessions.shape[0],chunk_size):
    # reset_index(drop=True)重建行索引，丢弃旧索引
    current_chunk=data.loc[sessions[i]:sessions[min(sessions.shape[0]-1,i+chunk_size-1)]].reset_index(drop=True)
    # 取出每个session后30个，考虑性能和最近行为
    # as_index保持原有的索引，不用session做索引
    current_chunk=current_chunk.groupby('session',as_index=False).nth(list(range(-30,0))).reset_index(drop=True)

    # 排除点击操作
    current_chunk=current_chunk[current_chunk['type'].isin([1,2])]

    # 按照session连接，对同一个session内的所有行（不区分属性）做笛卡尔积
    consecutive_AIDs=current_chunk.merge(current_chunk,on='session')

    # 排除自己和自己
    consecutive_AIDs=consecutive_AIDs[consecutive_AIDs.aid_x!=consecutive_AIDs.aid_y]

    # 通过时间戳(>=0)的方式保证两两算一次，不考虑顺序，保留14天内的结果
    consecutive_AIDs['days_elapsed']=(consecutive_AIDs.ts_y-consecutive_AIDs.ts_x)/(14*24*60*60)
    consecutive_AIDs=consecutive_AIDs[(consecutive_AIDs.days_elapsed>=0) & (consecutive_AIDs.days_elapsed<=1)]

    for aid_x,aid_y,type_y in zip(consecutive_AIDs['aid_x'],consecutive_AIDs['aid_y'],consecutive_AIDs['type_y']):
        next_AIDs[aid_x][aid_y]+=1

# 保存分数最高的15个
sub=[]
for aid_x, co_dict in next_AIDs.items():
    for aid_y, _ in co_dict.most_common(15):
        sub.append((aid_x, aid_y))
sub = pd.DataFrame(sub, columns=['aid_x','aid_y'])
sub.to_parquet('/home/mingyu/Recommand-System/项目/OTTO/save/co_visitation_result/buy.parquet', index=False)